# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoud-mos/my-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

# Pull a clean March 2026 sample into a Pandas DataFrame
query = f"""
SELECT 
    gsc_impressions,
    gsc_sum_position,
    sessions_ai,
    scroll_events,
    client_has_ga4,
    gsc_clicks 
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available = TRUE
LIMIT 10000;
"""
df = con.sql(query).df()

# Handle missing values and convert categoricals/booleans
df.fillna(0, inplace=True)
df['client_has_ga4'] = df['client_has_ga4'].astype(int)

print(f"Feature vector shape: {df.shape}")
print(df.head())

Feature vector shape: (10000, 6)
   gsc_impressions  gsc_sum_position  sessions_ai  scroll_events  \
0               20                67            0              0   
1                1                 0            0              0   
2              125               616            0              0   
3                7                28            0              0   
4               11                25            0              0   

   client_has_ga4  gsc_clicks  
0               0           0  
1               0           0  
2               0           1  
3               0           0  
4               0           0  


## 2. Feature notes (meaning, missing, categorical, available-when?)

- `gsc_impressions`: Total count of search impressions. Handled missing: filled with 0 (no impressions). Available: Pre-query.
- `gsc_sum_position`: Sum of rankings for impressions. Handled missing: filled with 0. Available: Pre-query.
- `sessions_ai`: AI-driven session count. Handled missing: filled with 0. Available: Pre-query.
- `scroll_events`: Total scroll events. Handled missing: filled with 0. Available: Pre-query.
- `client_has_ga4`: Boolean indicator for GA4 integration. Handled: cast to int (0/1). Available: Static context.
- `gsc_clicks`: Target variable. Available: Post-query (NOT usable as a feature).

gsc_impressions (Numeric): Total times the content was seen in search. Nulls filled with 0. recorded and available at the end of the day before the prediction window.

gsc_sum_position (Numeric): Aggregate ranking position. Nulls filled with 0. Available historically before prediction.

sessions_ai (Numeric): Traffic specifically driven by AI overviews. Nulls filled with 0. Historical totals are known before prediction.

scroll_events (Numeric): On-page user engagement depth. Nulls filled with 0. Captured in real-time, fully available before the next day's prediction.

client_has_ga4 (Categorical/Boolean): Whether the client has analytics configured. Mapped True/False to 1/0. This is a static configuration state known well before prediction time.

In [2]:
#check md cell 

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# The Trap: Engineer a leaked feature using the target variable (gsc_clicks)
df['accidental_ctr'] = df['gsc_clicks'] / (df['gsc_impressions'] + 1)

# Show the suspiciously high correlation (The "Skyrocket" effect)
correlation = df['accidental_ctr'].corr(df['gsc_clicks'])
print(
    f"Danger: Correlation between our leaked feature (CTR) and the target (Clicks) is {correlation:.4f}")
print("If this is close to 1.0, the model is cheating.\n")

# The Fix: Drop the leaked feature to keep the model honest
df_clean = df.drop(columns=['accidental_ctr'])
print(
    f"Leaked feature dropped. Clean columns ready for modeling: \n{list(df_clean.columns)}")

Danger: Correlation between our leaked feature (CTR) and the target (Clicks) is 0.1544
If this is close to 1.0, the model is cheating.

Leaked feature dropped. Clean columns ready for modeling: 
['gsc_impressions', 'gsc_sum_position', 'sessions_ai', 'scroll_events', 'client_has_ga4', 'gsc_clicks']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*
client_hash_id & content_hash_id: Excluded because they are unique identifiers. Feeding high-cardinality IDs to a model encourages it to memorize specific clients/pages rather than learning generalizable traffic patterns.

report_date: Excluded as a raw string/date to prevent time-series memorization. (If seasonality is needed later, it should be extracted into features like day_of_week).

gsc_data_available: Excluded because it is a system-level filtering flag, not a predictive human-behavior signal.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.